# Whakaari Cause–Trigger Analysis

This notebook applies the Cause–Trigger algorithm to the Whakaari 2019 eruption case. 

The full dataset is retained as the constructed data product, but the algorithm is applied only to a fixed case-study interval \(I\). The split into \(I_1\) and \(I_2\) is selected automatically from the effect variable.

## 1. Imports, paths, and case-study interval

In [ ]:
from pathlib import Path
import sys
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "whakaari"

for path in [SRC_DIR, NOTEBOOK_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

from cause_trigger_whakaari import *

EVENT_TIME = pd.Timestamp("2019-12-09 01:11:00", tz="UTC")
WHAKAARI_PATH = DATA_DIR / "whakaari_final.csv"

In [ ]:
X_full = load_model_frame(WHAKAARI_PATH)

CASE_START = EVENT_TIME - pd.Timedelta(days=14)
CASE_END = EVENT_TIME + pd.Timedelta(days=3)

X = X_full.loc[CASE_START:CASE_END].copy()

display(pd.DataFrame([model_overview(X, EFFECT)]))

,n_rows,n_variables,start,end,inferred_frequency,effect,effect_min,effect_median,effect_mean,effect_max,effect_abs_mean,effect_iqr,effect_n_unique
0,391,8,2019-11-25 02:00:00+00:00,2019-12-12 01:00:00+00:00,None,effect_tremor_5_15_scaled,-3.461211,0.683565,0.515423,4.561627,1.079183,0.552667,391


## 2. Reference lag selection

VAR-AIC and VAR-BIC are reported as automatic reference lags. The final interpretation is based on whether the result is stable across nearby lags in the 1–12 grid.

In [ ]:
MAX_LAGS = 12

reference_parameters = pd.DataFrame([
    select_parameters(
        X,
        EFFECT,
        max_lags=MAX_LAGS,
        criterion="aic",
        fallback_lag=1,
        fallback_distribution="gaussian",
    ),
    select_parameters(
        X,
        EFFECT,
        max_lags=MAX_LAGS,
        criterion="bic",
        fallback_lag=1,
        fallback_distribution="gaussian",
    ),
])

display(reference_parameters)

aic_row = reference_parameters.loc[
    reference_parameters["lag_method"] == "VAR-AIC"
].iloc[0]

selected_lag = int(aic_row["selected_lag"])
selected_distribution = aic_row["selected_distribution"]

c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


,effect,selected_lag,selected_distribution,lag_method,max_lags
0,effect_tremor_5_15_scaled,3,gaussian,VAR-AIC,6


## 3. Main configuration and automatic split

The split is selected automatically inside the fixed Whakaari case-study interval. The known eruption time is shown only as external context.

In [ ]:
workflow = WhakaariWorkflowConfig(
    effect=EFFECT,
    event_time=EVENT_TIME,
    alpha=0.05,
    selected_lag=selected_lag,
    max_lags=MAX_LAGS,
    min_I1_length=48,
    min_I2_length=30,
    distribution=selected_distribution,
    parameter_source="automatic_reference",
)

main_split = split_diagnostics(
    X,
    EFFECT,
    event_time=EVENT_TIME,
    min_I1_length=workflow.min_I1_length,
    min_I2_length=workflow.min_I2_length,
)

display(pd.DataFrame([main_split]))

plot_effect_with_split(
    X,
    EFFECT,
    main_split,
    event_time=workflow.event_time,
)

## 4. Reference-lag backend comparison

This run shows the result at the VAR-AIC reference lag. The result is considered stronger if the same pair also appears across adjacent lags in the stability grid.

In [ ]:
results, comparison, diagnostics = run_suite(
    X,
    workflow,
    lag=workflow.selected_lag,
    distribution=workflow.distribution,
    cond_ind_test="parcorr",
)

display(compact_comparison(comparison))
display(compact_diagnostics(diagnostics))

## 5. HMML lag-stability grid

The HMML backend is used as the baseline. A pair is considered stable if it appears over at least two adjacent lags.

In [ ]:
hmml_sensitivity = run_sensitivity_grid(
    X,
    workflow,
    run_specs=({"run": "hmml_backend_beta", "backend": "hmml"},),
    lags=range(1, workflow.max_lags + 1),
    distributions=(workflow.distribution,),
)

hmml_accepted = accepted_sensitivity_rows(hmml_sensitivity)
hmml_stability = summarise_lag_stability(
    hmml_sensitivity,
    backend="hmml",
    reference_lag=workflow.selected_lag,
    min_adjacent=2,
)

display(hmml_accepted)
display(hmml_stability)

In [ ]:
if not hmml_stability.empty and hmml_stability["stable"].any():
    representative_lag = int(
        hmml_stability.loc[hmml_stability["stable"], "recommended_lag"].iloc[0]
    )
    representative_pair = hmml_stability.loc[hmml_stability["stable"], "pair"].iloc[0]
else:
    representative_lag = workflow.selected_lag
    representative_pair = None

summary_choice = pd.DataFrame([{
    "reference_lag_VAR_AIC": workflow.selected_lag,
    "representative_lag": representative_lag,
    "representative_pair": representative_pair,
}])

display(summary_choice)

## 6. PCMCI lagged-link robustness

PCMCI is used as a stricter supplementary backend. This section uses only lagged PCMCI links and does not include contemporaneous trigger candidates.

In [ ]:
pcmci_sensitivity = run_sensitivity_grid(
    X,
    workflow,
    run_specs=({"run": "pcmci_ridge", "backend": "pcmci"},),
    lags=range(1, workflow.max_lags + 1),
    distributions=(workflow.distribution,),
    cond_ind_test="parcorr",
    use_contemporaneous_triggers=False,
)

display(compact_sensitivity(pcmci_sensitivity))
display(accepted_sensitivity_rows(pcmci_sensitivity))

## 7. PCMCI+ contemporaneous-trigger extension

PCMCI+ is used as a separate supplementary extension because it can estimate contemporaneous \(tau=0\) links. In this section, significant contemporaneous source-target links are allowed as immediate trigger candidates. They are not added to \(B_2\), and \(V\) is still constructed only from lagged \(B_2\) parents.

In [ ]:
pcmci_plus_tau0_sensitivity = run_sensitivity_grid(
    X,
    workflow,
    run_specs=({"run": "pcmci_plus_tau0_ridge", "backend": "pcmci_plus"},),
    lags=range(1, workflow.max_lags + 1),
    distributions=(workflow.distribution,),
    cond_ind_test="parcorr",
    use_contemporaneous_triggers=True,
)

display(compact_sensitivity(pcmci_plus_tau0_sensitivity))
display(accepted_sensitivity_rows(pcmci_plus_tau0_sensitivity))

In [ ]:
#final representative-lag PCMCI+ tau0 run
result_tau0, diag_tau0, row_tau0 = run_one(
    X,
    workflow,
    run_name="pcmci_plus_tau0_representative_lag",
    backend="pcmci_plus",
    lag=representative_lag,
    distribution=workflow.distribution,
    cond_ind_test="parcorr",
    parameter_source=f"stability_lag{representative_lag}_pcmci_plus_tau0_ridge",
    use_contemporaneous_triggers=True,
)

display(compact_comparison(pd.DataFrame([row_tau0])))
display(compact_diagnostics(diag_tau0))